<!-- AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07 -->

# Task 3: agentic equity research

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sathudeva7/CDAZZDEV-MLE-SATHURSAN/blob/main/task3_agentic/task3_agentic_research.ipynb)

This notebook is a thin driver. The logic lives in the `task3_agentic/` modules,
which reuse Task 1's data, indicators and news code, and carry offline tests in
`tests/`. Each section shows the evidence for one or more rubric rows. Results are
printed by code cells only, so no number in the text can drift from the run.

| Rubric row (marks) | Section | Where the logic lives |
|---|---|---|
| 3A All five tools implemented (15) | 3A.1 | `task3_agentic/tools.py`, `schemas.py` |
| 3A Autonomous tool selection (10) | 3A.2, 3A.5 (the order table) | `task3_agentic/agent.py` |
| 3A Observe and replan cycle (8) | 3A.2 (live trace), 3A.3 | `agent.py`, `printer.py` |
| 3A Final report quality (10) | 3A.4 | `task3_agentic/report.py` |
| 3A Error handling (7) | 3A.5 | `tools.py` (envelope + hint), `agent.py` |
| 3B Distinct roles and tool restriction (8) | 3B.3 | `pipeline.py`, `handoff.py` |
| 3B Structured handoff schema (8) | 3B.2 | `schemas.py` (`DataBrief`, `ClarificationRequest`, `ClarificationResponse`) |
| 3B Message trace visible (6) | 3B.1 | `printer.py` |
| 3B Critique loop (8) | 3B.1, 3B.2 | `pipeline.py`, `handoff.py` |
| 3B End-to-end automation (5) | 3B.1 | `pipeline.py` |
| 3C Short-term memory (5) | 3C.1 | `agent.py` (`InMemorySaver`, one thread per conversation) |
| 3C Persistent cache (5) | 3C.2 | `task3_agentic/cache.py` |
| 3C agent_trace.jsonl (5) | 3C.3 | `task3_agentic/trace.py` |

The query, from the brief: *Analyse the current financial health and market
sentiment of AAPL. Identify the top three risks to its share price over the next
90 days and suggest one data-driven hedge strategy.*

<!-- TESTING-ONLY(openai): rewrite this paragraph for the submission profile. -->
**Models.** This run uses the LLM profile named in the setup cell; its output
prints the profile and model. The agents' tool-calling model and every
structured answer (sentiment labels, the brief, the clarification request, the
reports) come from the same profile. Budgets per agent run: 8 tool calls and 6
model turns.

**To run it yourself:** add the keys the setup cell lists under Colab's Secrets
(the key icon), then Runtime → Run all.

## Setup

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
import os
import subprocess
import sys
import time
from pathlib import Path

RUN_STARTED = time.perf_counter()
TICKER = "AAPL"
BAD_TICKER = "NOTATICKERZZ"  # a symbol that does not exist, for the error-handling demos
REPO_URL = "https://github.com/sathudeva7/CDAZZDEV-MLE-SATHURSAN"
REPO_REF = "main"  # the branch or tag the Colab run clones
# Set here, not from the shell, so a local LLM_PROFILE cannot leak in.
NOTEBOOK_PROFILE = "openai_dev"  # TESTING-ONLY(openai): set to "free" before the submission run (CLAUDE.md rule 5)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content") / REPO_URL.rsplit("/", 1)[-1]
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "--quiet"], check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    # Locally the notebook runs from task3_agentic/, one level below the repo root.
    ROOT = Path.cwd().parent if Path.cwd().name == "task3_agentic" else Path.cwd()
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
sys.path.insert(0, str(ROOT))
os.environ["LLM_PROFILE"] = NOTEBOOK_PROFILE

from common.llm import enable_console_logging
from common.llm_config import active_profile, secret

enable_console_logging()
profile = active_profile()
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip()  # the hash is informational
print(f"Running in {'Colab' if IN_COLAB else 'a local kernel'}, repo at {commit or 'unknown commit'}")
print(f"LLM profile: {profile.name} (primary {profile.primary.name}: {profile.primary.model}; "
      f"fallback {profile.fallback.name + ': ' + profile.fallback.model if profile.fallback else 'none'})")
for provider in (profile.primary, profile.fallback):  # presence only, never the value
    if provider is not None:
        print(f"  {'✓' if secret(provider.api_key_env) else '✗'} {provider.api_key_env}")

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
# One run, one record: the trace, the call log and today's cache file are cleared,
# so the committed files match this notebook's outputs and the first 3B run is a real cache miss.
import json
from datetime import datetime

import pandas as pd
from IPython.display import Markdown, display

from task1_financial.pipeline import MARKET_TIMEZONE
from task3_agentic.cache import CACHE_DIR, cache_path
from task3_agentic.trace import DEFAULT_LOG_DIR, TRACE_FILE_NAME, read_trace

TODAY = datetime.now(MARKET_TIMEZONE).date()  # the market's date in New York, as every tool uses
TRACE = DEFAULT_LOG_DIR / TRACE_FILE_NAME
CALL_LOG = DEFAULT_LOG_DIR / profile.log_file
CACHE_FILE = cache_path(CACHE_DIR, TICKER, TODAY)
for path in (TRACE, CALL_LOG, CACHE_FILE):
    path.unlink(missing_ok=True)
    print("cleared", path.relative_to(ROOT))

RUN_LABELS: dict[str, str] = {}  # run_id -> section, for the trace summary in 3C.3


def tool_call_count() -> int:
    """tool_call lines written to agent_trace.jsonl so far, by every run."""
    return sum(event["event"] == "tool_call" for event in read_trace(TRACE))

## 3A.1 The five tools, called directly

Each tool returns the same envelope, `ToolResult`: `status` (`ok`, `empty` or
`error`), the typed `data`, `error`, `warnings`, and a `hint` naming an
alternative tool when the status is not ok. Tools never raise. Below, each one is
called by code, with no agent, on AAPL. `llm_sentiment` labels the headlines that
`get_news` returned.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.tools import ToolSession

direct = ToolSession(today=TODAY, subject=TICKER)
RUN_LABELS[direct.run_id] = "3A.1 direct tools"
results = {
    "get_price_data": direct.get_price_data(TICKER, "6mo"),
    "calculate_volatility": direct.calculate_volatility(TICKER, 30),
    "get_news": direct.get_news(TICKER, 10),
}
headlines = [h.title for h in results["get_news"].data.headlines] if results["get_news"].data else []
results["llm_sentiment"] = direct.llm_sentiment(headlines)
results["web_search"] = direct.web_search(f"{TICKER} stock analyst outlook next quarter")


def key_fact(result) -> str:
    """One line from each tool's typed data, read from its fields."""
    d = result.data
    if d is None:
        return result.error or "no data"
    if result.tool == "get_price_data":
        change = f"{d.period_return_pct:+.1f}%" if d.period_return_pct is not None else "n/a"
        return f"{len(d.bars)} bars to {d.as_of}, close {d.bars[-1].close:.2f}, 6-month return {change}"
    if result.tool == "calculate_volatility":
        return f"{d.window}-day vol {d.current_pct:.1f}% (1y median {d.median_1y_pct:.1f}%, percentile {d.percentile_1y:.0f})"
    if result.tool == "get_news":
        return f"{len(d.headlines)} headlines, newest: {d.headlines[0].title[:60]}" if d.headlines else "no headlines"
    if result.tool == "llm_sentiment":
        score = f"{d.summary.score:+.2f}" if d.summary.score is not None else "n/a"
        return f"Sentiment score {score} ({d.summary.label}), {d.summary.scored} scored"
    return f"{len(d.hits)} hits via the {d.backend} backend; first: {d.hits[0].title[:50]}" if d.hits else "no hits"


durations = {e["tool"]: e["duration_ms"] for e in read_trace(TRACE, run_id=direct.run_id) if e["event"] == "tool_call"}
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(
    [
        {"tool": name, "status": r.status, "data type": type(r.data).__name__, "key fact": key_fact(r), "duration_ms": durations.get(name)}
        for name, r in results.items()
    ]
)

One full envelope, as JSON. The agent never reads this whole object: it reads a
compact **digest** (`ToolResult.for_llm()`), shown second, which keeps a price
history of 120+ bars down to the latest values, period statistics and the last 5 bars.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
print(results["calculate_volatility"].model_dump_json(indent=2))
price = results["get_price_data"]
print(f"\nget_price_data: the typed result holds {len(price.data.bars)} bars; the digest the agent reads is {len(price.for_llm()):,} characters:")
print(price.for_llm()[:900], "...")

The argument schema each tool exposes to the model. Every tool also takes a
required `why`: the model's one-sentence reason for the call, which the trace
records. It is required because both models tested left the message text empty
when they called tools, so `why` is the only place their reasoning shows.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.tools import AGENT_TOOL_ARGS, TOOL_DESCRIPTIONS, TOOL_NAMES

for name in TOOL_NAMES:
    properties = AGENT_TOOL_ARGS[name].model_json_schema()["properties"]
    args = ", ".join(f"{arg}: {spec.get('type', 'enum' if 'enum' in spec else '?')}" for arg, spec in properties.items())
    print(f"{name}({args})")
    print(f"    {TOOL_DESCRIPTIONS[name][:150]}")

## 3A.2 The research agent

One agent with all five tools, built as a small LangGraph `StateGraph`:
`agent → tools → agent → … → finish`. The model decides which tools to call and
in what order. Nothing in the code fixes a sequence: the graph only loops until
the model stops calling tools or the budget (8 calls, 6 turns) runs out. Then the
`finish` step writes the report.

The trace below streams as the run goes. `[single] turn n` is a model turn, `->`
a tool it chose, with its `why`, and `ok` / `x` what came back.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.agent import MAX_TOOL_CALLS, ResearchAgent
from task3_agentic.printer import print_replan_cycles, print_update

session = ToolSession(today=TODAY, subject=TICKER)
RUN_LABELS[session.run_id] = "3A.2 research agent"
agent = ResearchAgent(session)
run = agent.research(TICKER, on_update=print_update)
main_order = [obs["tool"] for obs in run.observations]
print(f"\n{run.tool_calls} tool calls (budget {MAX_TOOL_CALLS}), thread {run.thread_id}")
print("order the model chose:", " -> ".join(main_order))

## 3A.3 Observe → replan

Read back from `agent_trace.jsonl`: each cycle pairs the tool results one model
turn received with the calls the **next** turn chose, and the reason it gave.
Calls made in the same turn were chosen together, so the cycle boundary is the turn.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
print_replan_cycles(read_trace(TRACE, run_id=session.run_id), agent.name)

## 3A.4 The final report

The report is written by one structured LLM call from the tool digests, then
**checked by code**. Evidence citing a tool that did not return a usable result
is dropped, with a warning. The hedge's price levels are computed by code (the
expected 90-day move = price × volatility × √(63/252), the SMA-200, the 52-week
low, the lower Bollinger band). The LLM only picks a strategy and the levels by
name. If the LLM fails, a template report is written from the same data.
`generated_by` says which happened.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
report = run.report
print(f"generated_by: {report.generated_by} | tools called: {report.tools_called} | warnings: {report.warnings or 'none'}")
display(Markdown(report.to_markdown()))

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
# The same report as the validated Pydantic object.
print(report.model_dump_json(indent=2))

## 3C.1 Short-term memory: a follow-up from context

The agent keeps each conversation in LangGraph's `InMemorySaver`, under a
`thread_id`. The follow-up goes to the same thread, so the earlier tool results
are still in its messages. Its tools stay bound, so whether it calls one is the
model's choice. The cell counts the trace's `tool_call` lines before and after.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
FOLLOW_UP = (
    "What annualised volatility did you find, and where does it sit in its one-year range? "
    "Answer from what you already retrieved."
)
before = tool_call_count()
follow_up = agent.ask(FOLLOW_UP, thread_id=run.thread_id, on_update=print_update)
new_lines = tool_call_count() - before
print(f"\nanswer: {follow_up.answer}")
print(f"\ntool calls in the follow-up: {follow_up.tool_calls}; new tool_call lines in agent_trace.jsonl: {new_lines}")
print("Answered from memory, without re-calling a tool." if new_lines == 0 else "The model re-called a tool for this follow-up (see the trace above).")

## 3A.5 Error handling

Three demos, from the tool level up:

1. **Tools on a bad ticker, no agent.** Each tool returns an `error` or `empty`
   envelope with a hint, and nothing raises.
2. **A tool that always fails.** `failing_tools=["get_news"]` is a demo switch,
   off by default, that makes `get_news` return an error. The agent reads the
   error and its hint, and tries another route.
3. **The agent on a bad ticker.** Every data tool fails. The run still ends with a
   report that names the gaps, rather than a traceback.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
bad = ToolSession(today=TODAY, subject=BAD_TICKER)
RUN_LABELS[bad.run_id] = "3A.5 bad ticker, direct"
bad_results = [
    bad.get_price_data(BAD_TICKER),
    bad.calculate_volatility(BAD_TICKER),
    bad.get_news(BAD_TICKER),
    bad.llm_sentiment([]),
    bad.web_search(f"{BAD_TICKER} stock"),
]
pd.DataFrame(
    [{"tool": r.tool, "status": r.status, "error / warnings": r.error or "; ".join(r.warnings) or "", "hint": r.hint} for r in bad_results]
)

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
failing = ToolSession(today=TODAY, subject=TICKER, failing_tools=["get_news"])
RUN_LABELS[failing.run_id] = "3A.5 get_news failing"
failing_run = ResearchAgent(failing).research(TICKER, on_update=print_update)
failing_order = [obs["tool"] for obs in failing_run.observations]
print()
print_replan_cycles(read_trace(TRACE, run_id=failing.run_id), "single")

**Same code, different order.** Both runs used the same agent class, prompt and
tools. Only what the tools returned differed. A hard-coded sequence would produce
the same column twice.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
rows = max(len(main_order), len(failing_order))
pad = lambda seq: seq + [""] * (rows - len(seq))
order = pd.DataFrame({"3A.2 main run": pad(main_order), "get_news failing": pad(failing_order)}, index=range(1, rows + 1))
order.index.name = "call"
print("failing run's statuses:", [obs["status"] for obs in failing_run.observations])
print("report data gaps:", failing_run.report.data_gaps)
order

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
bad_agent = ToolSession(today=TODAY, subject=BAD_TICKER)
RUN_LABELS[bad_agent.run_id] = "3A.5 bad ticker, agent"
bad_run = ResearchAgent(bad_agent).research(BAD_TICKER, on_update=print_update)
print("\ncalls:", [(obs["tool"], obs["status"]) for obs in bad_run.observations])
print("report generated_by:", bad_run.report.generated_by)
print("risks:", [risk.title for risk in bad_run.report.top_risks])
print("hedge:", bad_run.report.hedge.strategy, "| levels:", "none (no price data)" if bad_run.report.hedge.levels is None else "computed")
print("data gaps:")
for gap in bad_run.report.data_gaps:
    print(" -", gap)

## 3B.1 Two agents with a critique loop

A fixed outer graph coordinates two agents, each its own tool loop with its own
tools bound:

`cache_lookup → news_intake → Agent A → Agent B → Agent A answers → final_report → cache_save`

- **news_intake** (code, not an agent) fetches headlines for A, because A has no
  `get_news`.
- **Agent A, the data analyst** (`get_price_data`, `calculate_volatility`,
  `llm_sentiment`), produces a `DataBrief`. Code copies the numbers from A's tool
  results, and A's LLM writes 3–5 key observations.
- **Agent B, the research writer** (`web_search`, `get_news`), reads the brief,
  searches for commentary, and sends **one** `ClarificationRequest` back to A.
- **Agent A answers** with a `ClarificationResponse`, which holds the requested
  data as a typed snapshot. If A doesn't fetch it, the pipeline calls the tool
  for A and labels the answer `fulfilled_by: pipeline`.
- **final_report**: B writes the report from the brief, its own research and the
  answer, and says what the clarification changed.

The critique is a fixed edge, so it runs exactly once on every uncached run. No
step needs a person.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.pipeline import ResearchPipeline

pipeline_session = ToolSession(today=TODAY, subject=TICKER)
RUN_LABELS[pipeline_session.run_id] = "3B pipeline"
pipeline_started = time.perf_counter()
first = ResearchPipeline(pipeline_session).run(TICKER, on_update=print_update)
print(f"\nfinished in {time.perf_counter() - pipeline_started:.0f} s | cached: {first.cached} | warnings: {first.warnings or 'none'}")

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
print(f"generated_by: {first.report.generated_by} | tools called (both agents): {first.report.tools_called}")
display(Markdown(first.report.to_markdown()))

## 3B.2 The handoffs, as typed objects

Every message between the agents is a Pydantic model, validated on both sides
and written to the trace as JSON (`handoff` and `critique` events). Below are the
three objects from the run above.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.schemas import ClarificationRequest, ClarificationResponse, DataBrief

for obj in (first.brief, first.request, first.response):
    assert isinstance(obj, (DataBrief, ClarificationRequest, ClarificationResponse))
print("A -> B  DataBrief:")
print(first.brief.model_dump_json(indent=2))

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
print("B -> A  ClarificationRequest:")
print(first.request.model_dump_json(indent=2))
print("\nA -> B  ClarificationResponse (the request it answers is shown above):")
print(first.response.model_dump_json(indent=2, exclude={"request"}))
print("\nIncorporated: what B's final report says the clarification changed:")
print(first.report.clarification_used)

## 3B.3 Tool restriction

Restriction is enforced by **binding**: each agent's model is given only its own
tools, so it cannot even name another. If a model asks for a tool it wasn't
given, the tools node refuses the call and traces it as `refused`
(`tests/test_task3_pipeline.py::test_writer_cannot_call_price_tools`). The
crosstab counts the real run's traced calls by agent. `pipeline` is the news
intake step, which is code.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.handoff import ANALYST_TOOLS
from task3_agentic.pipeline import WRITER_TOOLS

print("Agent A is bound to:", ANALYST_TOOLS)
print("Agent B is bound to:", WRITER_TOOLS)
calls = pd.DataFrame([e for e in read_trace(TRACE, run_id=pipeline_session.run_id) if e["event"] == "tool_call"])
print("refused calls:", int((calls["status"] == "refused").sum()))
pd.crosstab(calls["agent"], calls["tool"])

## 3C.2 Persistent cache

After an uncached run, the pipeline saves the report, the brief and the
clarification exchange to `task3_agentic/cache/{TICKER}_{date}.json` (the
market's date in New York). Below, a **new** pipeline with a new tool session,
which stands in for a later run, finds the file, validates it against the
schema, and returns it without calling a single tool. A corrupt or old-schema
file counts as a miss. A template-written report is never cached.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
before = tool_call_count()
later_session = ToolSession(today=TODAY, subject=TICKER)
RUN_LABELS[later_session.run_id] = "3C.2 cache hit"
second = ResearchPipeline(later_session).run(TICKER, on_update=print_update)
new_lines = tool_call_count() - before
assert second.cached and new_lines == 0, (second.cached, new_lines)
assert second.report == first.report
print(f"cache hit: {second.cached} | new tool_call lines: {new_lines} | same report as the first run: {second.report == first.report}")
saved = json.loads(CACHE_FILE.read_text())
print(f"{CACHE_FILE.relative_to(ROOT)}: {CACHE_FILE.stat().st_size:,} bytes, keys {list(saved)}")
print({key: saved[key] for key in ("run_id", "profile", "created_at", "schema_version")})

## 3C.3 `agent_trace.jsonl`

Every tool call, whether made by code or by an agent, goes through
`ToolSession.call`, which writes one `tool_call` line: tool, args, the agent's
`why`, status, output (the digest, cut to 200 characters, with its full length
in `output_chars`), wall-clock `duration_ms`, and `cache_hit`. Model turns,
reports, handoffs, critiques and cache lookups are their own events. The
committed `task3_agentic/logs/agent_trace.jsonl` is this run's file.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
events = pd.DataFrame(read_trace(TRACE))
events["section"] = events["run_id"].map(RUN_LABELS)
print(f"{len(events)} events in {TRACE.relative_to(ROOT)}")
events.pivot_table(index="section", columns="event", values="ts", aggfunc="count", fill_value=0)

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
tool_calls = events[events["event"] == "tool_call"]
assert {"tool", "args", "output", "duration_ms"} <= set(tool_calls.columns)
print(f"{len(tool_calls)} tool calls; longest stored output: {tool_calls['output'].str.len().max()} characters")
tool_calls[["section", "agent", "tool", "args", "status", "duration_ms", "cache_hit", "output"]].head(12)

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
tool_calls.groupby("tool").agg(
    calls=("ts", "size"),
    ok=("status", lambda s: int((s == "ok").sum())),
    mean_ms=("duration_ms", "mean"),
    max_ms=("duration_ms", "max"),
    cache_hits=("cache_hit", "sum"),
).round(0)

## Run summary

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
calls_log = pd.read_json(CALL_LOG, lines=True) if CALL_LOG.exists() else pd.DataFrame()
llm_tokens = int(calls_log[["prompt_tokens", "completion_tokens"]].sum().sum()) if len(calls_log) else 0
agent_tokens = int(events.loc[events["event"] == "agent_turn", "tokens"].fillna(0).sum())
failed_at = next((i for i, obs in enumerate(failing_run.observations) if obs["tool"] == "get_news" and obs["status"] != "ok"), None)
switched = failing_order[failed_at + 1:] if failed_at is not None else []
print(f"Model:            {profile.name} ({profile.primary.model})")
print(f"3A report:        {report.generated_by}; risks: {[risk.title for risk in report.top_risks]}")
print(f"3A hedge:         {report.hedge.strategy}")
print(f"3A follow-up:     {follow_up.tool_calls} tool calls")
print(f"get_news failing: after the failure the agent called {switched}")
print(f"Bad ticker:       report by {bad_run.report.generated_by}, {len(bad_run.report.data_gaps)} data gaps, no exception")
print(f"3B request:       needs {first.request.needs}: {first.request.question}")
print(f"3B answer:        fulfilled by {first.response.fulfilled_by}; report by {first.report.generated_by}")
print(f"3C cache:         second run cached={second.cached}")
print(f"Trace:            { {event: int(count) for event, count in events['event'].value_counts().items()} }")
print(f"Tokens:           {agent_tokens:,} in agent turns + {llm_tokens:,} in structured calls ({len(calls_log)} logged)")
print(f"Wall time:        {time.perf_counter() - RUN_STARTED:.0f} s")

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
# Colab only: download the trace, the call log and the cache file, which are committed beside this notebook.
import zipfile

if IN_COLAB:
    from google.colab import files

    bundle = Path("/content/task3_run_outputs.zip")
    with zipfile.ZipFile(bundle, "w") as archive:
        for path in (TRACE, CALL_LOG, CACHE_FILE):
            if path.exists():
                archive.write(path, path.relative_to(ROOT))
        print("bundled:", archive.namelist())
    files.download(str(bundle))
else:
    print("Local run: the files are already in place:", [str(p.relative_to(ROOT)) for p in (TRACE, CALL_LOG, CACHE_FILE) if p.exists()])